# Enzyme Replicate Consistency Analysis

**Goal:** Characterize which BSH enzymes give reliable, reproducible signal across the 3 experimental replicates vs. which enzymes are noisy/inconsistent.

**Mirrors the amine consistency analysis** (`amine_replicate_consistency.ipynb`), but focuses on the **enzyme axis** instead of the product axis.

**Key questions:**
1. Which enzymes are the most/least consistent across replicates?
2. Is enzyme inconsistency concentrated in a few enzymes or spread evenly?
3. Do inconsistent enzymes tend to have borderline (weak) signal, or is noise random?
4. Is there a relationship between enzyme activity breadth and consistency?
5. How does enzyme consistency affect the intensity CV distribution?
6. Can we identify enzyme tiers (reliable, moderate, noisy) for model training?

**Data source:** Raw replicate-level intensities from:
- `NEW_Stage2_BAs_subs_for_heatmap_manual.csv` (12 substrate products)
- `NEW_Stage2_BAs_amines_for_heatmap_manual.csv` (82 amine products)

## Section 1: Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
RESULTS_DIR = OUTPUT_DIR / "model_outputs" / "enzyme_consistency"
RESULTS_DIR.mkdir(exist_ok=True, parents=True)

# Load both heatmap CSVs
df_subs = pd.read_csv(DATA_DIR / "NEW_Stage2_BAs_subs_for_heatmap_manual.csv")
df_amines = pd.read_csv(DATA_DIR / "NEW_Stage2_BAs_amines_for_heatmap_manual.csv")

# Merge
df_raw = pd.merge(df_subs, df_amines, on=['filename', 'Code', 'Replicate'], how='outer')
df_raw = df_raw[df_raw['Code'].notna() & (df_raw['Code'] != 'NA')].copy()

meta_cols = ['filename', 'Code', 'Replicate']
product_cols = [c for c in df_raw.columns if c not in meta_cols]

print(f"Enzymes: {df_raw['Code'].nunique()}")
print(f"Products: {len(product_cols)}")
print(f"Replicates: {sorted(df_raw['Replicate'].unique())}")
print(f"Total rows: {len(df_raw)}")

In [ ]:
# Parse product columns into (hydroxyl, amine)
def parse_product_col(col):
    parts = col.rsplit('_', 1)
    if len(parts) != 2 or not parts[1].isdigit():
        return None, None, None
    product_id = parts[1]
    remainder = parts[0]
    known_hydroxyls = ['3a,7a,12k', '3a7a12k', '3a12k', '3a7k', '3k12a', '3k7a', 'Di', 'Mono', 'Tri']
    for h in sorted(known_hydroxyls, key=len, reverse=True):
        if remainder.startswith(h + '_'):
            amine = remainder[len(h)+1:]
            return h, amine, product_id
    return None, None, None

product_info = {}
for col in product_cols:
    h, a, pid = parse_product_col(col)
    if h is not None:
        product_info[col] = {'hydroxyl': h, 'amine': a, 'product_id': pid}

print(f"Parsed {len(product_info)} / {len(product_cols)} product columns")
unique_amines = sorted(set(v['amine'] for v in product_info.values()))
unique_hydroxyls = sorted(set(v['hydroxyl'] for v in product_info.values()))
print(f"Unique amines: {len(unique_amines)}")
print(f"Unique hydroxyl patterns: {len(unique_hydroxyls)}")

## Section 2: Build Long-Format Replicate Stats

In [ ]:
# Melt to long format
df_long = df_raw.melt(
    id_vars=['Code', 'Replicate'],
    value_vars=list(product_info.keys()),
    var_name='product',
    value_name='intensity'
)
df_long['amine'] = df_long['product'].map(lambda x: product_info[x]['amine'])
df_long['hydroxyl'] = df_long['product'].map(lambda x: product_info[x]['hydroxyl'])
df_long['intensity'] = df_long['intensity'].fillna(0)
df_long['detected'] = (df_long['intensity'] > 0).astype(int)

# Per (enzyme, product) replicate stats
replicate_stats = df_long.groupby(['Code', 'product', 'amine', 'hydroxyl']).agg(
    n_reps=('intensity', 'count'),
    mean_intensity=('intensity', 'mean'),
    std_intensity=('intensity', 'std'),
    max_intensity=('intensity', 'max'),
    min_intensity=('intensity', 'min'),
    n_detected=('detected', 'sum'),
).reset_index()

replicate_stats['cv'] = np.where(
    replicate_stats['mean_intensity'] > 0,
    replicate_stats['std_intensity'] / replicate_stats['mean_intensity'],
    np.nan
)

def detection_category(row):
    if row['n_detected'] == 0:
        return 'never_detected'
    elif row['n_detected'] == row['n_reps']:
        return 'always_detected'
    else:
        return 'inconsistent'

replicate_stats['detection'] = replicate_stats.apply(detection_category, axis=1)

print(f"Total (enzyme, product) combos: {len(replicate_stats):,}")
print(f"\nOverall detection categories:")
print(replicate_stats['detection'].value_counts())

## Section 3: Per-Enzyme Detection Consistency Profile

In [ ]:
# For each enzyme, compute its consistency profile across ALL products
enzyme_detection = replicate_stats.groupby('Code').agg(
    n_products=('detection', 'count'),
    n_never=('detection', lambda x: (x == 'never_detected').sum()),
    n_always=('detection', lambda x: (x == 'always_detected').sum()),
    n_inconsistent=('detection', lambda x: (x == 'inconsistent').sum()),
).reset_index()

enzyme_detection['pct_never'] = enzyme_detection['n_never'] / enzyme_detection['n_products'] * 100
enzyme_detection['pct_always'] = enzyme_detection['n_always'] / enzyme_detection['n_products'] * 100
enzyme_detection['pct_inconsistent'] = enzyme_detection['n_inconsistent'] / enzyme_detection['n_products'] * 100
enzyme_detection['pct_consistent'] = (enzyme_detection['n_never'] + enzyme_detection['n_always']) / enzyme_detection['n_products'] * 100

# Activity breadth = products consistently detected
enzyme_detection['activity_breadth'] = enzyme_detection['n_always']
# Total detected (including inconsistent)
enzyme_detection['total_detected'] = enzyme_detection['n_always'] + enzyme_detection['n_inconsistent']

enzyme_detection = enzyme_detection.sort_values('pct_inconsistent', ascending=False)

print(f"Enzyme consistency profile ({len(enzyme_detection)} enzymes):")
print(f"\n  Mean % inconsistent: {enzyme_detection['pct_inconsistent'].mean():.1f}%")
print(f"  Median % inconsistent: {enzyme_detection['pct_inconsistent'].median():.1f}%")
print(f"  Range: {enzyme_detection['pct_inconsistent'].min():.1f}% – {enzyme_detection['pct_inconsistent'].max():.1f}%")
print(f"\n  Mean activity breadth (always detected): {enzyme_detection['activity_breadth'].mean():.1f} products")
print(f"  Max activity breadth: {enzyme_detection['activity_breadth'].max()} products")

print(f"\n--- Top 15 MOST INCONSISTENT enzymes ---")
print(enzyme_detection[['Code', 'n_always', 'n_inconsistent', 'n_never', 'pct_inconsistent', 'pct_consistent']].head(15).to_string(index=False))

print(f"\n--- Top 15 MOST CONSISTENT enzymes ---")
print(enzyme_detection.sort_values('pct_inconsistent')[['Code', 'n_always', 'n_inconsistent', 'n_never', 'pct_inconsistent', 'pct_consistent']].head(15).to_string(index=False))

## Section 4: Enzyme Consistency Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# (A) Histogram: % inconsistent per enzyme
ax = axes[0, 0]
ax.hist(enzyme_detection['pct_inconsistent'], bins=25, color='#e74c3c', edgecolor='black', linewidth=0.5, alpha=0.8)
ax.axvline(enzyme_detection['pct_inconsistent'].mean(), color='black', linestyle='--', 
           label=f"Mean: {enzyme_detection['pct_inconsistent'].mean():.1f}%")
ax.axvline(enzyme_detection['pct_inconsistent'].median(), color='blue', linestyle='--', 
           label=f"Median: {enzyme_detection['pct_inconsistent'].median():.1f}%")
ax.set_xlabel('% of Products with Inconsistent Detection', fontsize=11)
ax.set_ylabel('Number of Enzymes', fontsize=11)
ax.set_title('(A) Distribution of Enzyme Inconsistency Rate', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# (B) Histogram: activity breadth
ax = axes[0, 1]
ax.hist(enzyme_detection['activity_breadth'], bins=25, color='#2ecc71', edgecolor='black', linewidth=0.5, alpha=0.8)
ax.axvline(enzyme_detection['activity_breadth'].mean(), color='black', linestyle='--',
           label=f"Mean: {enzyme_detection['activity_breadth'].mean():.1f}")
ax.set_xlabel('Number of Products Always Detected (3/3 reps)', fontsize=11)
ax.set_ylabel('Number of Enzymes', fontsize=11)
ax.set_title('(B) Enzyme Activity Breadth (Consistent Products)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# (C) Scatter: activity breadth vs inconsistency count
ax = axes[1, 0]
ax.scatter(enzyme_detection['activity_breadth'], enzyme_detection['n_inconsistent'],
           alpha=0.6, s=40, c='steelblue', edgecolors='black', linewidth=0.3)
ax.set_xlabel('Products Always Detected (3/3)', fontsize=11)
ax.set_ylabel('Products Inconsistently Detected', fontsize=11)
ax.set_title('(C) Activity Breadth vs. Inconsistency Count', fontsize=12)
ax.grid(True, alpha=0.2)
# Add correlation
r, p = stats.pearsonr(enzyme_detection['activity_breadth'], enzyme_detection['n_inconsistent'])
ax.annotate(f'r = {r:.2f}, p = {p:.2e}', xy=(0.05, 0.95), xycoords='axes fraction',
            fontsize=10, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.5))

# (D) Stacked bar: consistent vs inconsistent fraction per enzyme (sorted)
ax = axes[1, 1]
df_sorted = enzyme_detection.sort_values('pct_inconsistent', ascending=True)
x = range(len(df_sorted))
ax.bar(x, df_sorted['pct_always'], color='#2ecc71', label='Always detected', width=1.0)
ax.bar(x, df_sorted['pct_inconsistent'], bottom=df_sorted['pct_always'], 
       color='#e74c3c', label='Inconsistent', width=1.0)
ax.bar(x, df_sorted['pct_never'], 
       bottom=df_sorted['pct_always'] + df_sorted['pct_inconsistent'],
       color='#bdc3c7', label='Never detected', width=1.0)
ax.set_xlabel('Enzymes (sorted by inconsistency)', fontsize=11)
ax.set_ylabel('% of Products', fontsize=11)
ax.set_title('(D) Consistency Profile per Enzyme', fontsize=12)
ax.legend(fontsize=9, loc='upper left')
ax.set_xlim(-1, len(df_sorted))
ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'enzyme_consistency_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: enzyme_consistency_distribution.png")

## Section 5: Ranked Enzyme Consistency (Barh Chart)

Equivalent to the amine stacked bar chart — one bar per enzyme, showing what fraction of its products are always/never/inconsistently detected.

In [ ]:
# Top 40 most inconsistent enzymes (horizontal stacked bar)
df_plot = enzyme_detection.sort_values('pct_inconsistent', ascending=False).head(40)

fig, ax = plt.subplots(figsize=(12, 14))
y_pos = range(len(df_plot))

ax.barh(y_pos, df_plot['pct_always'].values, color='#2ecc71', label='Always detected (3/3)', 
        edgecolor='white', linewidth=0.5)
ax.barh(y_pos, df_plot['pct_inconsistent'].values, left=df_plot['pct_always'].values, 
        color='#e74c3c', label='Inconsistent (1/3 or 2/3)', edgecolor='white', linewidth=0.5)
ax.barh(y_pos, df_plot['pct_never'].values, 
        left=(df_plot['pct_always'] + df_plot['pct_inconsistent']).values,
        color='#bdc3c7', label='Never detected (0/3)', edgecolor='white', linewidth=0.5)

ax.set_yticks(y_pos)
ax.set_yticklabels(df_plot['Code'].values, fontsize=8)
ax.set_xlabel('% of Products', fontsize=12)
ax.set_title('Top 40 Most Inconsistent Enzymes\n(detection consistency across 3 replicates)', fontsize=13)
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim(0, 100)
ax.grid(True, alpha=0.2, axis='x')

# Add inconsistency % labels
for i, (_, row) in enumerate(df_plot.iterrows()):
    if row['pct_inconsistent'] > 3:
        ax.text(row['pct_always'] + row['pct_inconsistent']/2, i,
                f"{row['pct_inconsistent']:.0f}%", va='center', ha='center', 
                fontsize=7, color='white', fontweight='bold')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'enzyme_ranked_inconsistency.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: enzyme_ranked_inconsistency.png")

## Section 6: Per-Enzyme Intensity CV Analysis

For products where an enzyme shows activity, how reproducible is the **intensity** (not just detection)?

In [ ]:
# Filter to products detected at least once
df_detected = replicate_stats[replicate_stats['n_detected'] > 0].copy()

# Per-enzyme CV stats
enzyme_cv = df_detected.groupby('Code').agg(
    n_detected_products=('cv', 'count'),
    median_cv=('cv', 'median'),
    mean_cv=('cv', 'mean'),
    q25_cv=('cv', lambda x: x.quantile(0.25)),
    q75_cv=('cv', lambda x: x.quantile(0.75)),
    median_intensity=('mean_intensity', 'median'),
    mean_intensity=('mean_intensity', 'mean'),
    pct_high_cv=('cv', lambda x: (x > 1.0).mean() * 100),  # % products with CV > 1.0
).reset_index().sort_values('median_cv')

print(f"Enzyme-level CV summary ({len(enzyme_cv)} enzymes with detected products):")
print(f"\n  Median CV across enzymes: {enzyme_cv['median_cv'].median():.3f}")
print(f"  Range of median CVs: {enzyme_cv['median_cv'].min():.3f} – {enzyme_cv['median_cv'].max():.3f}")
print(f"\n--- Top 15 MOST REPRODUCIBLE enzymes (lowest median CV) ---")
print(enzyme_cv[['Code', 'n_detected_products', 'median_cv', 'mean_cv', 'pct_high_cv', 'median_intensity']].head(15).to_string(index=False))

print(f"\n--- Top 15 NOISIEST enzymes (highest median CV) ---")
print(enzyme_cv.sort_values('median_cv', ascending=False)[['Code', 'n_detected_products', 'median_cv', 'mean_cv', 'pct_high_cv', 'median_intensity']].head(15).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# (A) Histogram of median CV per enzyme
ax = axes[0, 0]
ax.hist(enzyme_cv['median_cv'], bins=25, color='steelblue', edgecolor='black', linewidth=0.5, alpha=0.8)
ax.axvline(enzyme_cv['median_cv'].median(), color='red', linestyle='--',
           label=f"Median: {enzyme_cv['median_cv'].median():.3f}")
ax.axvline(1.0, color='orange', linestyle='--', alpha=0.7, label='CV = 1.0')
ax.set_xlabel('Median CV Across Detected Products', fontsize=11)
ax.set_ylabel('Number of Enzymes', fontsize=11)
ax.set_title('(A) Distribution of Per-Enzyme Intensity Reproducibility', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# (B) Scatter: median CV vs number of detected products
ax = axes[0, 1]
ax.scatter(enzyme_cv['n_detected_products'], enzyme_cv['median_cv'],
           alpha=0.6, s=40, c='steelblue', edgecolors='black', linewidth=0.3)
ax.set_xlabel('Number of Detected Products', fontsize=11)
ax.set_ylabel('Median CV', fontsize=11)
ax.set_title('(B) Activity Breadth vs. Intensity Reproducibility', fontsize=12)
ax.axhline(1.0, color='orange', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.2)
r, p = stats.pearsonr(enzyme_cv['n_detected_products'], enzyme_cv['median_cv'])
ax.annotate(f'r = {r:.2f}, p = {p:.2e}', xy=(0.05, 0.95), xycoords='axes fraction',
            fontsize=10, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.5))

# (C) Scatter: median intensity vs median CV (is low signal = high noise?)
ax = axes[1, 0]
ax.scatter(np.log10(enzyme_cv['median_intensity'] + 1), enzyme_cv['median_cv'],
           alpha=0.6, s=40, c='darkorange', edgecolors='black', linewidth=0.3)
ax.set_xlabel('log10(Median Intensity + 1)', fontsize=11)
ax.set_ylabel('Median CV', fontsize=11)
ax.set_title('(C) Signal Strength vs. Reproducibility', fontsize=12)
ax.axhline(1.0, color='orange', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.2)
r, p = stats.pearsonr(np.log10(enzyme_cv['median_intensity'] + 1), enzyme_cv['median_cv'])
ax.annotate(f'r = {r:.2f}, p = {p:.2e}', xy=(0.05, 0.95), xycoords='axes fraction',
            fontsize=10, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.5))

# (D) Histogram: % of products with CV > 1.0 per enzyme
ax = axes[1, 1]
ax.hist(enzyme_cv['pct_high_cv'], bins=25, color='#e74c3c', edgecolor='black', linewidth=0.5, alpha=0.8)
ax.set_xlabel('% of Detected Products with CV > 1.0', fontsize=11)
ax.set_ylabel('Number of Enzymes', fontsize=11)
ax.set_title('(D) Distribution of High-Noise Product Fraction', fontsize=12)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'enzyme_cv_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: enzyme_cv_analysis.png")

## Section 7: Enzyme Consistency Box Plots

CV distribution per enzyme (like the amine CV box plot, but per enzyme).

In [ ]:
# Box plot of CV per enzyme — show top 30 most inconsistent + top 30 most consistent
# (all enzymes would be too many to display)

# Identify top/bottom enzymes by median CV
cv_sorted = enzyme_cv.sort_values('median_cv')
top_consistent = cv_sorted.head(20)['Code'].tolist()
top_noisy = cv_sorted.tail(20)['Code'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: most consistent
ax = axes[0]
df_box = df_detected[df_detected['Code'].isin(top_consistent)]
order = cv_sorted[cv_sorted['Code'].isin(top_consistent)].sort_values('median_cv')['Code'].tolist()
sns.boxplot(data=df_box, x='Code', y='cv', order=order, color='#2ecc71', fliersize=2, ax=ax)
ax.set_xlabel('Enzyme', fontsize=11)
ax.set_ylabel('CV (across 3 replicates)', fontsize=11)
ax.set_title('20 Most Consistent Enzymes (lowest median CV)', fontsize=12)
ax.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='CV = 1.0')
ax.legend(fontsize=9)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
ax.grid(True, alpha=0.2, axis='y')

# Right: noisiest
ax = axes[1]
df_box = df_detected[df_detected['Code'].isin(top_noisy)]
order = cv_sorted[cv_sorted['Code'].isin(top_noisy)].sort_values('median_cv')['Code'].tolist()
sns.boxplot(data=df_box, x='Code', y='cv', order=order, color='#e74c3c', fliersize=2, ax=ax)
ax.set_xlabel('Enzyme', fontsize=11)
ax.set_ylabel('CV (across 3 replicates)', fontsize=11)
ax.set_title('20 Noisiest Enzymes (highest median CV)', fontsize=12)
ax.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='CV = 1.0')
ax.legend(fontsize=9)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'enzyme_cv_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: enzyme_cv_boxplots.png")

## Section 8: Enzyme Consistency Tiers

Classify enzymes into reliability tiers based on their overall consistency profile.

In [ ]:
# Merge detection consistency with CV data
enzyme_full = enzyme_detection.merge(enzyme_cv, on='Code', how='left')

# Define tiers based on % inconsistent AND median CV
def assign_tier(row):
    pct_inc = row['pct_inconsistent']
    med_cv = row['median_cv'] if pd.notna(row['median_cv']) else 0
    
    if pct_inc <= 5 and med_cv <= 0.5:
        return 'Tier 1: Highly Reliable'
    elif pct_inc <= 15 and med_cv <= 0.8:
        return 'Tier 2: Moderate'
    elif pct_inc <= 25:
        return 'Tier 3: Somewhat Noisy'
    else:
        return 'Tier 4: Very Noisy'

enzyme_full['tier'] = enzyme_full.apply(assign_tier, axis=1)

print("Enzyme Reliability Tiers:")
tier_counts = enzyme_full['tier'].value_counts().sort_index()
for tier, count in tier_counts.items():
    pct = count / len(enzyme_full) * 100
    print(f"  {tier}: {count} enzymes ({pct:.1f}%)")

print(f"\n--- Tier statistics ---")
tier_stats = enzyme_full.groupby('tier').agg(
    n_enzymes=('Code', 'count'),
    mean_pct_inconsistent=('pct_inconsistent', 'mean'),
    mean_median_cv=('median_cv', 'mean'),
    mean_activity_breadth=('activity_breadth', 'mean'),
    mean_total_detected=('total_detected', 'mean'),
).sort_index()
print(tier_stats.to_string())

In [ ]:
# Scatter plot colored by tier
tier_colors = {
    'Tier 1: Highly Reliable': '#2ecc71',
    'Tier 2: Moderate': '#3498db',
    'Tier 3: Somewhat Noisy': '#f39c12',
    'Tier 4: Very Noisy': '#e74c3c',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: % inconsistent vs median CV, colored by tier
ax = axes[0]
for tier, color in tier_colors.items():
    mask = enzyme_full['tier'] == tier
    subset = enzyme_full[mask]
    ax.scatter(subset['pct_inconsistent'], subset['median_cv'],
              alpha=0.7, s=50, c=color, edgecolors='black', linewidth=0.3,
              label=f"{tier} (n={len(subset)})")
ax.set_xlabel('% Products with Inconsistent Detection', fontsize=11)
ax.set_ylabel('Median CV of Detected Products', fontsize=11)
ax.set_title('Enzyme Reliability: Detection Consistency vs. Intensity Variability', fontsize=12)
ax.legend(fontsize=8, loc='upper left')
ax.grid(True, alpha=0.2)

# Right: activity breadth vs % inconsistent, colored by tier
ax = axes[1]
for tier, color in tier_colors.items():
    mask = enzyme_full['tier'] == tier
    subset = enzyme_full[mask]
    ax.scatter(subset['activity_breadth'], subset['pct_inconsistent'],
              alpha=0.7, s=50, c=color, edgecolors='black', linewidth=0.3,
              label=f"{tier} (n={len(subset)})")
ax.set_xlabel('Activity Breadth (Products Always Detected)', fontsize=11)
ax.set_ylabel('% Products with Inconsistent Detection', fontsize=11)
ax.set_title('Enzyme Activity Breadth vs. Inconsistency', fontsize=12)
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'enzyme_reliability_tiers.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: enzyme_reliability_tiers.png")

## Section 9: Per-Enzyme x Amine Consistency Heatmap

Is enzyme inconsistency driven by specific amines, or is it genome-wide?

In [ ]:
# Per (enzyme, amine): fraction of products that are inconsistent
enz_amine_incons = replicate_stats.groupby(['Code', 'amine']).agg(
    n_products=('detection', 'count'),
    n_inconsistent=('detection', lambda x: (x == 'inconsistent').sum()),
    n_always=('detection', lambda x: (x == 'always_detected').sum()),
).reset_index()
enz_amine_incons['pct_inconsistent'] = enz_amine_incons['n_inconsistent'] / enz_amine_incons['n_products'] * 100

# Pivot: enzyme x amine
incons_pivot = enz_amine_incons.pivot_table(
    index='Code', columns='amine', values='pct_inconsistent', fill_value=0
)

# Sort enzymes by overall inconsistency, amines by overall inconsistency
enz_order = enzyme_detection.sort_values('pct_inconsistent', ascending=False)['Code']
amine_incons = enz_amine_incons.groupby('amine')['pct_inconsistent'].mean().sort_values(ascending=False)
amine_order = amine_incons.index

incons_pivot = incons_pivot.reindex(index=enz_order, columns=amine_order)

# Show top 40 enzymes for readability
fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    incons_pivot.iloc[:40],
    cmap='YlOrRd',
    vmin=0, vmax=60,
    linewidths=0.3, linecolor='white',
    cbar_kws={'label': '% Inconsistent Detection'},
    ax=ax,
    xticklabels=True,
    yticklabels=True,
)
ax.set_xlabel('Amine', fontsize=12)
ax.set_ylabel('Enzyme (sorted by overall inconsistency)', fontsize=12)
ax.set_title('Enzyme x Amine Inconsistency Rate (Top 40 Noisiest Enzymes)', fontsize=13)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.setp(ax.get_yticklabels(), fontsize=7)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'enzyme_amine_inconsistency_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: enzyme_amine_inconsistency_heatmap.png")

## Section 10: Per-Replicate Enzyme Profiles

Do certain enzymes show systematically different behavior in one replicate vs. another?

In [ ]:
# For each enzyme, compute total intensity and number of detected products per replicate
rep_profiles = df_long.groupby(['Code', 'Replicate']).agg(
    total_intensity=('intensity', 'sum'),
    n_detected=('detected', 'sum'),
    mean_intensity_detected=('intensity', lambda x: x[x > 0].mean() if (x > 0).any() else 0),
).reset_index()

# Pivot for pairwise comparison
rep_pivot_intensity = rep_profiles.pivot_table(
    index='Code', columns='Replicate', values='total_intensity'
).reset_index()

rep_pivot_detected = rep_profiles.pivot_table(
    index='Code', columns='Replicate', values='n_detected'
).reset_index()

# Check for enzymes where replicate detection counts differ a lot
rep_pivot_detected['range'] = rep_pivot_detected[['rep1', 'rep2', 'rep3']].max(axis=1) - rep_pivot_detected[['rep1', 'rep2', 'rep3']].min(axis=1)
rep_pivot_detected['mean_detected'] = rep_pivot_detected[['rep1', 'rep2', 'rep3']].mean(axis=1)
rep_pivot_detected['cv_detected'] = rep_pivot_detected[['rep1', 'rep2', 'rep3']].std(axis=1) / rep_pivot_detected['mean_detected']

print("Per-replicate detection count variability:")
print(f"  Mean range (max-min detected products): {rep_pivot_detected['range'].mean():.1f}")
print(f"  Median range: {rep_pivot_detected['range'].median():.0f}")
print(f"  Max range: {rep_pivot_detected['range'].max():.0f}")

print(f"\n--- Enzymes with largest replicate-to-replicate detection difference ---")
top_diff = rep_pivot_detected.sort_values('range', ascending=False).head(15)
print(top_diff[['Code', 'rep1', 'rep2', 'rep3', 'range', 'cv_detected']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

rep_cols = ['rep1', 'rep2', 'rep3']
pairs = [('rep1', 'rep2'), ('rep1', 'rep3'), ('rep2', 'rep3')]
titles = ['Rep1 vs Rep2', 'Rep1 vs Rep3', 'Rep2 vs Rep3']

for ax, (r1, r2), title in zip(axes, pairs, titles):
    ax.scatter(rep_pivot_detected[r1], rep_pivot_detected[r2],
              alpha=0.5, s=30, c='steelblue', edgecolors='black', linewidth=0.3)
    maxval = max(rep_pivot_detected[r1].max(), rep_pivot_detected[r2].max())
    ax.plot([0, maxval], [0, maxval], 'r--', alpha=0.5, label='y=x')
    ax.set_xlabel(f'{r1}: # products detected', fontsize=11)
    ax.set_ylabel(f'{r2}: # products detected', fontsize=11)
    ax.set_title(f'{title} (detected product count)', fontsize=12)
    r_val = rep_pivot_detected[r1].corr(rep_pivot_detected[r2])
    ax.annotate(f'r = {r_val:.3f}', xy=(0.05, 0.95), xycoords='axes fraction',
                fontsize=10, va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.5))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

plt.suptitle('Per-Enzyme Replicate Agreement: Number of Detected Products', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'enzyme_replicate_pairwise.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: enzyme_replicate_pairwise.png")

## Section 11: Enzyme Consistency vs. Amine Consistency Cross-Comparison

Are the inconsistent enzyme-product combos driven more by the enzyme or the amine?

In [ ]:
# For each inconsistent (enzyme, product) combo, tag with enzyme tier and amine
inconsistent_combos = replicate_stats[replicate_stats['detection'] == 'inconsistent'].copy()

# Merge enzyme tier
tier_map = enzyme_full.set_index('Code')['tier'].to_dict()
inconsistent_combos['enzyme_tier'] = inconsistent_combos['Code'].map(tier_map)

# How much inconsistency is explained by the enzyme vs. the amine?
print("=== INCONSISTENCY ATTRIBUTION ===")
print(f"\nTotal inconsistent combos: {len(inconsistent_combos)}")

# By enzyme tier
print(f"\n--- By Enzyme Tier ---")
tier_incons = inconsistent_combos['enzyme_tier'].value_counts().sort_index()
tier_total = enzyme_full['tier'].value_counts().sort_index()
for tier in tier_incons.index:
    n_inc = tier_incons[tier]
    n_enz = tier_total[tier]
    print(f"  {tier}: {n_inc} inconsistent combos across {n_enz} enzymes ({n_inc/n_enz:.1f} per enzyme)")

# By amine
print(f"\n--- By Amine (top 10) ---")
amine_incons_counts = inconsistent_combos['amine'].value_counts().head(10)
for amine, count in amine_incons_counts.items():
    print(f"  {amine:30s} {count:4d} inconsistent combos")

# Variance decomposition: how much of the inconsistency signal is enzyme-driven vs amine-driven?
# Use a simple approach: for each inconsistent combo, compute mean inconsistency rate of its enzyme and amine
enzyme_incons_rate = enzyme_detection.set_index('Code')['pct_inconsistent']
amine_incons_rate = enz_amine_incons.groupby('amine')['pct_inconsistent'].mean()

inconsistent_combos['enzyme_incons_rate'] = inconsistent_combos['Code'].map(enzyme_incons_rate)
inconsistent_combos['amine_incons_rate'] = inconsistent_combos['amine'].map(amine_incons_rate)

print(f"\n--- Correlation of Inconsistency with Enzyme vs. Amine Rates ---")
# For ALL combos (not just inconsistent), does the enzyme or amine better predict inconsistency?
all_combos = replicate_stats.copy()
all_combos['is_inconsistent'] = (all_combos['detection'] == 'inconsistent').astype(int)
all_combos['enzyme_incons_rate'] = all_combos['Code'].map(enzyme_incons_rate)
all_combos['amine_incons_rate'] = all_combos['amine'].map(amine_incons_rate)

from sklearn.metrics import mutual_info_score

# Point-biserial correlation (binary outcome vs continuous predictor)
r_enzyme, p_enzyme = stats.pointbiserialr(all_combos['is_inconsistent'], all_combos['enzyme_incons_rate'])
r_amine, p_amine = stats.pointbiserialr(all_combos['is_inconsistent'], all_combos['amine_incons_rate'])

print(f"  Enzyme inconsistency rate: r = {r_enzyme:.3f} (p = {p_enzyme:.2e})")
print(f"  Amine inconsistency rate:  r = {r_amine:.3f} (p = {p_amine:.2e})")
print(f"\n  Interpretation: {'Enzyme' if abs(r_enzyme) > abs(r_amine) else 'Amine'} is a stronger predictor of inconsistency")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: inconsistency count by tier and amine
ax = axes[0]
tier_amine = inconsistent_combos.groupby(['enzyme_tier', 'amine']).size().unstack(fill_value=0)
# Collapse to just the tier totals for a cleaner bar chart
tier_totals = inconsistent_combos.groupby('enzyme_tier').size().sort_index()
colors_tier = [tier_colors.get(t, 'gray') for t in tier_totals.index]
bars = ax.barh(range(len(tier_totals)), tier_totals.values, color=colors_tier, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(tier_totals)))
ax.set_yticklabels(tier_totals.index, fontsize=9)
ax.set_xlabel('Number of Inconsistent Combos', fontsize=11)
ax.set_title('Inconsistency by Enzyme Tier', fontsize=12)
ax.grid(True, alpha=0.2, axis='x')
for i, (val, tier) in enumerate(zip(tier_totals.values, tier_totals.index)):
    n_enz = tier_total[tier]
    ax.text(val + 10, i, f"{val/n_enz:.1f}/enzyme", va='center', fontsize=9)

# Right: enzyme vs amine inconsistency rate for all combos
ax = axes[1]
# Compute mean inconsistency rate per (enzyme, amine) pair
summary = enz_amine_incons.copy()
summary['enzyme_overall'] = summary['Code'].map(enzyme_incons_rate)
summary['amine_overall'] = summary['amine'].map(amine_incons_rate)

ax.scatter(summary['enzyme_overall'], summary['amine_overall'],
           alpha=0.3, s=15, c='steelblue', edgecolors='none')
ax.set_xlabel('Enzyme Overall Inconsistency Rate (%)', fontsize=11)
ax.set_ylabel('Amine Overall Inconsistency Rate (%)', fontsize=11)
ax.set_title('Enzyme vs. Amine Inconsistency Rate\n(each dot = one enzyme-amine pair)', fontsize=12)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'enzyme_vs_amine_inconsistency.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: enzyme_vs_amine_inconsistency.png")

## Section 12: Borderline Signal Analysis

Are inconsistent enzymes producing **weak borderline signal** (near detection limit) or is the inconsistency random?

In [ ]:
# For each enzyme, compare the intensity of consistent vs inconsistent products
consistent_intensities = replicate_stats[replicate_stats['detection'] == 'always_detected']['mean_intensity']
inconsistent_intensities = replicate_stats[replicate_stats['detection'] == 'inconsistent']['mean_intensity']

print("=== SIGNAL STRENGTH: CONSISTENT vs. INCONSISTENT PRODUCTS ===")
print(f"\nAlways detected (consistent):")
print(f"  n = {len(consistent_intensities)}")
print(f"  Median intensity: {consistent_intensities.median():.0f}")
print(f"  Mean intensity: {consistent_intensities.mean():.0f}")
print(f"  25th-75th percentile: {consistent_intensities.quantile(0.25):.0f} – {consistent_intensities.quantile(0.75):.0f}")

print(f"\nInconsistently detected:")
print(f"  n = {len(inconsistent_intensities)}")
print(f"  Median intensity: {inconsistent_intensities.median():.0f}")
print(f"  Mean intensity: {inconsistent_intensities.mean():.0f}")
print(f"  25th-75th percentile: {inconsistent_intensities.quantile(0.25):.0f} – {inconsistent_intensities.quantile(0.75):.0f}")

# Mann-Whitney U test
u_stat, p_val = stats.mannwhitneyu(consistent_intensities, inconsistent_intensities, alternative='greater')
print(f"\nMann-Whitney U test (consistent > inconsistent): U = {u_stat:.0f}, p = {p_val:.2e}")
print(f"Effect size (rank-biserial r): {1 - 2*u_stat/(len(consistent_intensities)*len(inconsistent_intensities)):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: log-intensity distribution by detection category
ax = axes[0]
for cat, color, label in [('always_detected', '#2ecc71', 'Always detected'),
                           ('inconsistent', '#e74c3c', 'Inconsistent')]:
    vals = replicate_stats[replicate_stats['detection'] == cat]['mean_intensity']
    vals_log = np.log10(vals + 1)
    ax.hist(vals_log, bins=40, alpha=0.5, color=color, label=label, density=True, edgecolor='black', linewidth=0.3)
ax.set_xlabel('log10(Mean Intensity + 1)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Signal Strength: Consistent vs. Inconsistent Products', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)

# Right: per-enzyme, compare median intensity of consistent vs inconsistent products
ax = axes[1]
enz_consistent = replicate_stats[replicate_stats['detection'] == 'always_detected'].groupby('Code')['mean_intensity'].median()
enz_inconsistent = replicate_stats[replicate_stats['detection'] == 'inconsistent'].groupby('Code')['mean_intensity'].median()

# Align on enzymes that have both
common_enzymes = enz_consistent.index.intersection(enz_inconsistent.index)
ax.scatter(np.log10(enz_consistent[common_enzymes] + 1), 
           np.log10(enz_inconsistent[common_enzymes] + 1),
           alpha=0.6, s=40, c='steelblue', edgecolors='black', linewidth=0.3)
maxval = max(np.log10(enz_consistent[common_enzymes] + 1).max(), 
             np.log10(enz_inconsistent[common_enzymes] + 1).max())
ax.plot([0, maxval], [0, maxval], 'r--', alpha=0.5, label='y=x')
ax.set_xlabel('log10(Median Intensity of Consistent Products + 1)', fontsize=10)
ax.set_ylabel('log10(Median Intensity of Inconsistent Products + 1)', fontsize=10)
ax.set_title('Per-Enzyme: Consistent vs. Inconsistent Signal Strength', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'borderline_signal_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: borderline_signal_analysis.png")

## Section 13: Summary & Save

In [ ]:
# Comprehensive enzyme summary table
enzyme_summary = enzyme_full[[
    'Code', 'n_products', 'n_always', 'n_inconsistent', 'n_never',
    'pct_always', 'pct_inconsistent', 'pct_never', 'pct_consistent',
    'activity_breadth', 'total_detected',
    'n_detected_products', 'median_cv', 'mean_cv', 'pct_high_cv',
    'median_intensity', 'tier'
]].copy()

# Add per-replicate detection counts
rep_merge = rep_pivot_detected[['Code', 'rep1', 'rep2', 'rep3', 'range', 'cv_detected']].copy()
rep_merge.columns = ['Code', 'rep1_detected', 'rep2_detected', 'rep3_detected', 'rep_range', 'rep_cv']
enzyme_summary = enzyme_summary.merge(rep_merge, on='Code', how='left')

enzyme_summary = enzyme_summary.sort_values('pct_inconsistent', ascending=False)

print(f"Enzyme Summary Table: {len(enzyme_summary)} enzymes")
print(f"\nColumn descriptions:")
print(f"  n_products: total products tested")
print(f"  n_always/n_inconsistent/n_never: detection category counts")
print(f"  pct_*: percentages of each category")
print(f"  activity_breadth: products always detected (3/3)")
print(f"  median_cv: median intensity CV across detected products")
print(f"  pct_high_cv: % of detected products with CV > 1.0")
print(f"  tier: reliability tier assignment")
print(f"  rep*_detected: products detected per replicate")
print(f"  rep_range: max-min detected products across replicates")

print(f"\n--- First 20 rows ---")
cols_display = ['Code', 'n_always', 'n_inconsistent', 'pct_inconsistent', 'median_cv', 'pct_high_cv', 'tier']
print(enzyme_summary[cols_display].head(20).to_string(index=False))

In [ ]:
# Save all outputs
enzyme_summary.to_csv(RESULTS_DIR / 'enzyme_consistency_summary.csv', index=False)
enzyme_cv.to_csv(RESULTS_DIR / 'enzyme_cv_stats.csv', index=False)
enz_amine_incons.to_csv(RESULTS_DIR / 'enzyme_amine_inconsistency.csv', index=False)
rep_pivot_detected.to_csv(RESULTS_DIR / 'enzyme_replicate_detection_counts.csv', index=False)

print("Saved CSVs:")
for f in sorted(RESULTS_DIR.glob("*.csv")):
    print(f"  {f.name}")
print("\nSaved PNGs:")
for f in sorted(RESULTS_DIR.glob("*.png")):
    print(f"  {f.name}")

In [ ]:
print("=" * 80)
print("ENZYME REPLICATE CONSISTENCY -- SUMMARY")
print("=" * 80)

n_enz = len(enzyme_detection)
print(f"\nData: {n_enz} enzymes x {enzyme_detection['n_products'].iloc[0]} products x 3 replicates")

print(f"\n--- Detection Consistency (per enzyme) ---")
print(f"  Mean % inconsistent products per enzyme: {enzyme_detection['pct_inconsistent'].mean():.1f}%")
print(f"  Median: {enzyme_detection['pct_inconsistent'].median():.1f}%")
print(f"  Range: {enzyme_detection['pct_inconsistent'].min():.1f}% – {enzyme_detection['pct_inconsistent'].max():.1f}%")

print(f"\n--- Intensity Variability ---")
print(f"  Median enzyme CV: {enzyme_cv['median_cv'].median():.3f}")
print(f"  Range: {enzyme_cv['median_cv'].min():.3f} – {enzyme_cv['median_cv'].max():.3f}")

print(f"\n--- Reliability Tiers ---")
for tier in sorted(enzyme_full['tier'].unique()):
    n = (enzyme_full['tier'] == tier).sum()
    print(f"  {tier}: {n} ({n/n_enz*100:.0f}%)")

print(f"\n--- Key Findings ---")

# Most inconsistent
worst = enzyme_detection.sort_values('pct_inconsistent', ascending=False).head(3)
print(f"  Most inconsistent enzymes:")
for _, row in worst.iterrows():
    print(f"    {row['Code']}: {row['pct_inconsistent']:.1f}% inconsistent")

# Most consistent
best = enzyme_detection.sort_values('pct_inconsistent').head(3)
print(f"  Most consistent enzymes:")
for _, row in best.iterrows():
    print(f"    {row['Code']}: {row['pct_inconsistent']:.1f}% inconsistent")

print(f"\n  Enzyme-amine inconsistency attribution:")
print(f"    Enzyme rate correlation with inconsistency: r = {r_enzyme:.3f}")
print(f"    Amine rate correlation with inconsistency:  r = {r_amine:.3f}")
print(f"    -> {'Enzyme' if abs(r_enzyme) > abs(r_amine) else 'Amine'} identity is a stronger predictor")

print(f"\n--- Implications for Model Training ---")
n_tier1 = (enzyme_full['tier'] == 'Tier 1: Highly Reliable').sum()
n_tier4 = (enzyme_full['tier'] == 'Tier 4: Very Noisy').sum()
print(f"  1. {n_tier1} enzymes are highly reliable — cleanest training signal")
print(f"  2. {n_tier4} enzymes are very noisy — contribute uncertain labels")
print(f"  3. Consider sample weighting by enzyme reliability tier")
print(f"  4. Inconsistent products tend to have {'weaker' if consistent_intensities.median() > inconsistent_intensities.median() else 'comparable'} signal")
print(f"     (median consistent: {consistent_intensities.median():.0f} vs inconsistent: {inconsistent_intensities.median():.0f})")
print(f"  5. Enzyme hold-out CV should monitor if noisy enzymes hurt generalization")

print("\nDone!")